In [11]:
import eurostat
import pandas as pd
import numpy as np
import requests, json

%load_ext autoreload
%autoreload 2

DEMOGRAPH = "demo_r_d2jan"
ENVIRONMENT = "env_air_gge"
DBS = [DEMOGRAPH, ENVIRONMENT]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
data = eurostat.get_data_df(DEMOGRAPH)
data.to_csv("demo.csv", index=False)
data = eurostat.get_data_df(ENVIRONMENT)
data.to_csv("env.csv", index=False)

# Dimensions and label meanings

In [ ]:
for DB in DBS:
    print(DB)

    dims = eurostat.get_pars(DB)
    print(f"Dimension labels: {dims}\n")

    for dim in dims:
        print(f"{dim}'s label meanings:\n{eurostat.get_dic(DB, dim)}\n")


In [ ]:
df = eurostat.get_data_df(DEMOGRAPH)
#print(df.columns)

geo_data_demo = sorted(df["geo\\TIME_PERIOD"].unique())
geo_demo_nuts2 = sorted({g for g in geo_data_demo if len(g) == 4 and not g.endswith("ZZ") and not g.endswith("XX") and g not in {"EU28", "EFTA"}})
print(f"Demographic's NUTS2 region labels: {geo_demo_nuts2}")

df = eurostat.get_data_df(ENVIRONMENT)
#print(df.columns)

geo_data_env = sorted(df["geo\\TIME_PERIOD"].unique())
geo_env_country = sorted(set([g for g in geo_data_env if len(g) == 2]))
print(f"Environmental emission country labels: {geo_env_country}")

In [ ]:
from datetime import datetime

df = eurostat.get_data_df(DEMOGRAPH)
dim = eurostat.get_pars(DEMOGRAPH)
demo_slice = df[(df["age"] == "TOTAL")
                & (df["sex"] == "T")
                & (df["geo\\TIME_PERIOD"].isin(geo_demo_nuts2))]
demo_vals = demo_slice[demo_slice.columns[len(dim):]]
print(f"Demographic's value range: {demo_vals.min().min()}, {demo_vals.max().max()}")

df_env = eurostat.get_data_df(ENVIRONMENT)
dim_env = eurostat.get_pars(ENVIRONMENT)
AGG = {"EU27_2020", "EU28", "EA19", "EA20"}
env_slice = df_env[(df_env["airpol"] == "GHG")
                   & (df_env["src_crf"] == "TOTX4_MEMO")
                   & (df_env["geo\\TIME_PERIOD"].isin(geo_env_country))]
env_vals = env_slice[env_slice.columns[len(dim_env):]]
print(f"Environmental emission value range: {env_vals.min().min()}, {env_vals.max().max()}")

print(f"Demographic's data staleness: {datetime.now().year - int(demo_slice.columns[len(dim):].max())} year(s)")
print(f"Environmental emission data staleness: {datetime.now().year - int(env_slice.columns[len(dim_env):].max())} year(s)")

Demographic's value range: 0.0, 15907951.0
Environmental emission value range: 1.83974, 1253127.92
Demographic's data staleness: 1 year(s)
Environmental emission data staleness: 2 year(s)


In [ ]:
BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
resp = requests.get(f"{BASE}/demo_r_d2jan",
    params={"format":"JSON","lang":"en","age":"TOTAL","sex":"T"},
    timeout=30)
resp.raise_for_status()
d = resp.json()
print(json.dumps(d, indent=2))
# print(d["value"])
# print(d["dimension"]["time"]["category"]["index"])

In [ ]:
# keys = np.array([36])
# np.unravel_index(keys, [1,1,1,1,521,36])
# # -> (array([0,0,0]), ..., array([0,1,468]), array([0,0,0]))
# print(36//521)
# print(521//36)
# print(36%521)
# print(521%36)

resp = requests.get(f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/env_air_gge",
    params={"format":"JSON", "lang":"en"},
    timeout=30)
resp.raise_for_status()
d = resp.json()
print(json.dumps(d, indent=2))

keys = np.array([int(k) for k in d["value"].keys()])
print(keys)
inds = np.unravel_index(keys, d["size"])
print(inds)

cols = {}
for name, ind in zip(d["id"], inds):
    inv = {int(v): k for k, v in d["dimension"][name]["category"]["index"].items()}
    cols[name] = [inv[i] for i in ind]
cols["value"] = [float(v) for v in d["value"].values()]

data = pd.DataFrame(cols)
data.head()

# freq_inv = {int(v): k for k, v in d["dimension"]["freq"]["category"]["index"].items()}
# unit_inv = {int(v): k for k, v in d["dimension"]["unit"]["category"]["index"].items()}
# sex_inv = {int(v): k for k, v in d["dimension"]["sex"]["category"]["index"].items()}
# age_inv = {int(v): k for k, v in d["dimension"]["age"]["category"]["index"].items()}
# geo_inv = {int(v): k for k, v in d["dimension"]["geo"]["category"]["index"].items()}
# time_inv = {int(v): k for k, v in d["dimension"]["time"]["category"]["index"].items()}

In [17]:
from eurostat_dq.ingest import fetch_dataset_json
from eurostat_dq.config import DATASETS

df = fetch_dataset_json("env_air_gge", use_cache=True)
df = fetch_dataset_json("demo_r_d2jan", geo="HU11", sex="T", age="TOTAL")
df.shape
df.head()

DataFrame found in cache
DataFrame acquired from the internet
DataFrame saved to cache


,freq,unit,sex,age,geo,time,value
0,A,NR,T,TOTAL,HU11,2001,1759209.0
1,A,NR,T,TOTAL,HU11,2002,1739569.0
2,A,NR,T,TOTAL,HU11,2003,1719342.0
3,A,NR,T,TOTAL,HU11,2004,1705309.0
4,A,NR,T,TOTAL,HU11,2005,1697343.0


In [32]:
from eurostat_dq.ingest import fetch_dataset
from eurostat_dq.config import DATASETS
import numpy as np

df = fetch_dataset("demo_r_d2jan", use_cache=True)
df = df.rename(columns={"geo\\TIME_PERIOD":"geo"})

cols = list(df.columns)
base_dims = [dim for dim in cols if not str(dim).isdigit()]

df_long = (df.melt(id_vars=base_dims, var_name="time", value_name="value").dropna(subset=["value"]))
df_long["time"] = df_long["time"].astype(int)

print(df.shape)
print(df.head(20))

df_long.shape
df_long.head()

DataFrame found in cache
['freq', 'unit', 'sex', 'age', 'geo', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
[]
['freq', 'unit', 'sex', 'age', 'geo']
(160800, 41)
   freq unit sex    age   geo       1990       1991       1992       1993  \
0     A   NR   F  TOTAL    AL        NaN        NaN        NaN        NaN   
1     A   NR   F  TOTAL   AL0        NaN        NaN        NaN        NaN   
2     A   NR   F  TOTAL  AL01        NaN        NaN        NaN        NaN   
3     A   NR   F  TOTAL  AL02        NaN        NaN        NaN        NaN   
4     A   NR   F  TOTAL  AL03        NaN        NaN        NaN        NaN   
5     A   NR   F  TOTAL   ALX        NaN        NaN        NaN        NaN   
6     A   NR   F  TOTAL  ALXX        NaN        NaN     

,freq,unit,sex,age,geo,time,value
7,A,NR,F,TOTAL,AT,1990,3989903.0
8,A,NR,F,TOTAL,AT1,1990,1700829.0
9,A,NR,F,TOTAL,AT11,1990,139775.0
10,A,NR,F,TOTAL,AT12,1990,752424.0
11,A,NR,F,TOTAL,AT13,1990,808630.0


In [12]:
from eurostat_dq.ingest import fetch_dataset, fetch_dataset_json
from eurostat_dq.clean import to_tidy, apply_slice
from eurostat_dq.config import DATASETS

print("FETCH:")
for code, cfg in DATASETS.items():
    raw   = fetch_dataset(code, use_cache=False)
    tidy  = to_tidy(raw)
    clean = apply_slice(tidy, cfg)
    print(code, clean.head)

print("FETCH JSON:")
for code, cfg in DATASETS.items():
    raw   = fetch_dataset_json(code, use_cache=False)
    tidy  = to_tidy(raw)
    clean = apply_slice(tidy, cfg)
    print(code, clean.head())

FETCH:
DataFrame acquired from the internet
DataFrame saved to cache
demo_r_d2jan <bound method NDFrame.head of         freq unit sex    age   geo  time      value
107209     A   NR   T  TOTAL  AT11  1990   270670.0
107210     A   NR   T  TOTAL  AT12  1990  1455968.0
107211     A   NR   T  TOTAL  AT13  1990  1492636.0
107213     A   NR   T  TOTAL  AT21  1990   544983.0
107214     A   NR   T  TOTAL  AT22  1990  1169578.0
...      ...  ...  ..    ...   ...   ...        ...
5735661    A   NR   T  TOTAL  TRB1  2025  1724320.0
5735662    A   NR   T  TOTAL  TRB2  2025  2152387.0
5735664    A   NR   T  TOTAL  TRC1  2025  2961139.0
5735665    A   NR   T  TOTAL  TRC2  2025  4071429.0
5735666    A   NR   T  TOTAL  TRC3  2025  2457718.0

[10320 rows x 7 columns]>
DataFrame acquired from the internet
DataFrame saved to cache
env_air_gge <bound method NDFrame.head of         freq   unit airpol     src_crf geo  time         value
16487      A  MIO_T    GHG  TOTX4_MEMO  AT  1990      79.66846
42996  

In [ ]:
print(df)

        freq unit sex     age   geo  time      value
0          A   NR   F   TOTAL    AL  2000  1526762.0
1          A   NR   F   TOTAL    AL  2001  1535822.0
2          A   NR   F   TOTAL    AL  2002  1532563.0
3          A   NR   F   TOTAL    AL  2003  1526180.0
4          A   NR   F   TOTAL    AL  2004  1520481.0
...      ...  ...  ..     ...   ...   ...        ...
4671906    A   NR   T  Y_OPEN  UKN0  2014      250.0
4671907    A   NR   T  Y_OPEN  UKN0  2015      265.0
4671908    A   NR   T  Y_OPEN  UKN0  2016      274.0
4671909    A   NR   T  Y_OPEN  UKN0  2017      281.0
4671910    A   NR   T  Y_OPEN  UKN0  2018      282.0

[4671911 rows x 7 columns]
        freq   unit    airpol   src_crf geo  time      value
0          A  MIO_T       CH4      CRF1  AT  1990    0.05442
1          A  MIO_T       CH4      CRF1  AT  1991    0.04970
2          A  MIO_T       CH4      CRF1  AT  1992    0.04877
3          A  MIO_T       CH4      CRF1  AT  1993    0.04635
4          A  MIO_T       CH4  

In [51]:
from eurostat_dq.config import DATASETS, PROJECT_ROOT
from eurostat_dq.ingest import fetch_dataset
from eurostat_dq.clean import to_tidy

df = fetch_dataset(ENVIRONMENT, use_cache=True)
df = to_tidy(df)

filters = DATASETS[ENVIRONMENT].filters
geo_level = DATASETS[ENVIRONMENT].geo_level.lower()

geo_data = sorted(df["geo"].unique())
geo_data_leveled = []

if geo_level == "nuts2":
    geo_data_leveled = sorted({g for g in geo_data if len(g) == 4 and not g.endswith(("ZZ", "XX")) and not g.startswith(("EU", "EA")) and not g.isalpha()})
    print(f"Data of NUTS2 region labels: {geo_data_leveled}")
elif geo_level == "country":
    geo_data_leveled = sorted(set([g for g in geo_data if len(g) == 2]))
    print(f"Data of Country labels: {geo_data_leveled}")

mask = pd.Series(True, index=df.index)
mask &= df["geo"].isin(geo_data_leveled)
for col, val in filters.items():
    mask &= df[col] == val
df = df[mask]
cache_path = PROJECT_ROOT / "data" / "processed" / f"{ENVIRONMENT}.parquet"
cache_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(cache_path, index=False)
df.shape
df.head()

# print("DATA:")
# for i in {"EU27_2020", "EU28", "EA19", "EA20"}:
#     print(i)
#     print(df[df["geo"] == i].head(1))


DataFrame found in cache
Data of Country labels: ['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'TR']


,freq,unit,airpol,src_crf,geo,time,value
16487,A,MIO_T,GHG,TOTX4_MEMO,AT,1990,79.66846
16488,A,MIO_T,GHG,TOTX4_MEMO,BE,1990,145.46665
16489,A,MIO_T,GHG,TOTX4_MEMO,BG,1990,100.19699
16490,A,MIO_T,GHG,TOTX4_MEMO,CH,1990,55.16536
16491,A,MIO_T,GHG,TOTX4_MEMO,CY,1990,5.57948
